# Lab 10 - Containerizing an OpenAI-Compatible API

**Production Readiness Pack | Local Docker recommended**

This lab packages a Lab 5-style FastAPI proxy as a standard Docker image. Even if you run the notebook in Colab where Docker is usually unavailable, you will produce the exact files a DevOps/platform team expects.

## Learning Objectives

1. Generate a real OpenAI-compatible FastAPI server.
2. Package it with `requirements.txt`, `.dockerignore`, and a non-root `Dockerfile`.
3. Understand runtime secrets and environment variables.
4. Build and run the image locally when Docker is available.
5. Map the same image to Cloud Run, App Runner, ECS, or Kubernetes.

## Why Containers Matter For LLM Deployment

A notebook proves the idea. A container proves the runtime can be recreated.

When a platform team asks for a Docker image, they are asking for answers to practical deployment questions:

- Which Python version and dependencies are required?
- How does the service start?
- Which port does it listen on?
- How are secrets injected?
- Can the platform restart it and check health automatically?
- Can the same image run in staging and production with different environment variables?

The LLM part is only one piece. The deployment unit must behave like normal software.

## 1. The Files A Container Needs

A minimal production API image needs:

- Application code: `server.py`
- Dependencies: `requirements.txt`
- Build recipe: `Dockerfile`
- Exclusions: `.dockerignore`

Never bake `.env`, API keys, notebooks, vector databases, or local caches into the image unless you have a deliberate reason.

In [ ]:
%%writefile server.py
import os
from typing import Any, Dict, List, Optional

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from openai import OpenAI
from pydantic import BaseModel, Field

MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o-mini")
UPSTREAM_BASE_URL = os.getenv("UPSTREAM_BASE_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

app = FastAPI(title="OpenAI-Compatible LLM API", version="1.0.0")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = None
    messages: List[ChatMessage]
    temperature: float = 0.2
    stream: bool = False
    max_tokens: Optional[int] = Field(default=None, ge=1)

def upstream_client() -> OpenAI:
    if not OPENAI_API_KEY:
        raise HTTPException(status_code=500, detail="OPENAI_API_KEY is not configured")
    kwargs: Dict[str, Any] = {"api_key": OPENAI_API_KEY}
    if UPSTREAM_BASE_URL:
        kwargs["base_url"] = UPSTREAM_BASE_URL
    return OpenAI(**kwargs)

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "upstream_base_url": UPSTREAM_BASE_URL or "https://api.openai.com/v1"}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": MODEL_NAME, "object": "model", "owned_by": "classroom-api"}]}

@app.post("/v1/chat/completions")
def chat_completions(request: ChatCompletionRequest):
    client = upstream_client()
    model = request.model or MODEL_NAME
    messages = [m.model_dump() for m in request.messages]

    if request.stream:
        def event_stream():
            stream = client.chat.completions.create(model=model, messages=messages, temperature=request.temperature, max_tokens=request.max_tokens, stream=True)
            for chunk in stream:
                yield "data: " + chunk.model_dump_json() + "\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(event_stream(), media_type="text/event-stream")

    response = client.chat.completions.create(model=model, messages=messages, temperature=request.temperature, max_tokens=request.max_tokens)
    return response.model_dump()


In [ ]:
%%writefile requirements.txt
fastapi==0.115.6
uvicorn[standard]==0.34.0
openai>=1.57.0
pydantic>=2.10.0

In [ ]:
%%writefile .dockerignore
.env
.env.*
__pycache__/
*.pyc
*.ipynb
.ipynb_checkpoints/
chroma_db/
vector_db/
mlflow.db
mlruns/
.git/
.DS_Store

## Aha Moment: Image Build Time vs Container Runtime

A common beginner mistake is putting secrets or environment-specific settings into the image. The image should contain code and dependencies. The container runtime should provide secrets and configuration.

| Belongs in the image | Belongs at runtime |
| --- | --- |
| `server.py` | `OPENAI_API_KEY` |
| Python dependencies | `MODEL_NAME` |
| Startup command | `UPSTREAM_BASE_URL` |
| Non-secret defaults | Client API keys, tenant config, cloud secrets |

This separation lets the same image run against OpenAI, Azure OpenAI, vLLM, or LiteLLM simply by changing environment variables.

## 2. Dockerfile

This image runs as a non-root user and reads secrets at runtime. That is the correct default for cloud deployment.

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1     PYTHONUNBUFFERED=1

WORKDIR /app

RUN useradd --create-home --shell /bin/bash appuser

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY server.py .

USER appuser
EXPOSE 8000

CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"]

## 3. Local Smoke Test Without Docker

You can test the server directly first. In a terminal:

```bash
export OPENAI_API_KEY=your_key
uvicorn server:app --host 0.0.0.0 --port 8000
```

Then in another terminal or notebook cell:

```python
from openai import OpenAI
client = OpenAI(api_key="local-dev", base_url="http://localhost:8000/v1")
print(client.models.list())
```

In [ ]:
from pathlib import Path
for file_name in ["server.py", "requirements.txt", ".dockerignore", "Dockerfile"]:
    path = Path(file_name)
    print(f"{file_name}: {path.stat().st_size} bytes")

## 4. Build And Run With Docker

Run these commands on a local machine with Docker installed:

```bash
docker build -t llm-api:lab10 .
docker run --rm -p 8000:8000   -e OPENAI_API_KEY="$OPENAI_API_KEY"   -e MODEL_NAME="gpt-4o-mini"   llm-api:lab10
```

If your upstream is vLLM, Ollama, LiteLLM, or another OpenAI-compatible server, add:

```bash
-e UPSTREAM_BASE_URL="https://your-upstream.example.com/v1"
```

In [ ]:
# Optional local check. In Colab this often prints that Docker is unavailable.
!docker --version || true

## 5. OpenAI SDK Smoke Test

After the container is running, this client code should work without knowing your server is FastAPI.

In [ ]:
smoke_test_code = """
from openai import OpenAI

client = OpenAI(api_key="not-used-by-local-proxy", base_url="http://localhost:8000/v1")
print(client.models.list())

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello from the containerized API."}],
)
print(response.choices[0].message.content)
"""
print(smoke_test_code)

## Production Gotchas

Containerization is necessary for many production environments, but it is not the whole deployment story.

- A FastAPI proxy does not make model inference scalable by itself. For self-hosted models, you still need vLLM, TGI, Ollama, or another inference runtime.
- A container filesystem is usually ephemeral. Do not assume local ChromaDB files survive restarts unless you mount a persistent volume or use a managed vector database.
- Health checks should be cheap. `/health` should not call the LLM provider on every probe.
- Logs should go to stdout/stderr so the cloud platform can collect them.
- Concurrency matters. A single container can receive multiple requests at once, so avoid mutable global state unless it is thread-safe.
- Secrets should come from Secret Manager, AWS Secrets Manager, Kubernetes Secrets, or platform environment variables.

The course pattern is: notebook -> API -> container -> cloud runtime -> observability/eval loop.

## 6. Cloud Handoff

The same image can go to several platforms:

| Platform | Good For | Notes |
| --- | --- | --- |
| Cloud Run | Simple stateless HTTP APIs | Easy autoscaling, good first cloud target |
| AWS App Runner | Simple containerized web services | Less infrastructure than ECS |
| ECS/Fargate | Production services on AWS | More networking/IAM control |
| Kubernetes | Multi-service platforms | Powerful but operationally heavier |
| vLLM container | High-throughput model serving | Use when you host the model itself |

Production checklist:

- Secrets injected at runtime, never copied into image.
- Logs go to stdout/stderr.
- `/health` works without calling the expensive model.
- Container is stateless.
- Image has pinned dependencies.
- Autoscaling and concurrency limits are configured by the platform.

## Student Exercise

1. Add an `X-Request-ID` header to the FastAPI response.
2. Add a simple API-key check for clients calling your proxy.
3. Explain where provider secrets live in your cloud platform.
4. Decide: would your Capstone run better as HF Spaces, Docker API, or vLLM deployment?

## Key Takeaways

- Docker packages your API as a standard deployment unit.
- The model provider key is a runtime secret, not source code.
- OpenAI-compatible APIs are easy to smoke test after packaging.
- Containers solve packaging, but production still needs auth, observability, rate limits, and cost controls.